# MFI: Modern Fortran Interfaces — GPU/CuBLAS Test

**Repo:** https://github.com/14NGiestas/mfi/tree/impl/cublas

Sets up the Fortran + CUDA toolchain on Colab, clones `impl/cublas`, builds with cuBLAS, and runs tests.

**Runtime:** Select **GPU** (T4) before running.

In [ ]:
# 1. Verify GPU is available
!nvidia-smi

In [ ]:
# 2. Install Fortran toolchain + BLAS/LAPACK
!apt-get update -qq && apt-get install -y -qq \
    gfortran libblas-dev liblapack-dev libopenblas-dev \
    python3-pip 2>&1 | tail -3
!pip install -q fypp

# Install fpm 0.13.0 (supports profiles)
!curl -sL https://github.com/fortran-lang/fpm/releases/download/v0.13.0/fpm-0.13.0-linux-x86_64-gcc-12 \
    -o /usr/local/bin/fpm && chmod +x /usr/local/bin/fpm

!echo '--- versions ---'
!gfortran --version | head -1
!fpm --version
!fypp --version

In [ ]:
# 3. Install CUDA Toolkit (headers + libraries)
# Colab has the GPU driver but NOT the CUDA Toolkit dev headers.
# We install the toolkit so cublas_wrap.c can find cuda_runtime.h.
import subprocess, os, glob

# Check if CUDA headers already exist
header_found = glob.glob("/usr/local/cuda*/include/cuda_runtime.h")
if not header_found:
    print("Installing CUDA Toolkit (this may take a minute)...")
    # Ubuntu package that provides cuda_runtime.h and libcublas
    result = subprocess.run(
        ["apt-get", "install", "-y", "-qq", "nvidia-cuda-toolkit"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        # Fallback: install the runtime + dev packages separately
        print("nvidia-cuda-toolkit failed, trying alternative packages...")
        subprocess.run(["apt-get", "install", "-y", "-qq",
                        "libcublas-dev", "libcudart-dev"],
                       capture_output=True, text=True)
    header_found = glob.glob("/usr/local/cuda*/include/cuda_runtime.h")
    if not header_found:
        header_found = glob.glob("/usr/include/cuda_runtime.h")

if header_found:
    include_dir = os.path.dirname(header_found[0])
    cuda_base = os.path.dirname(include_dir)  # e.g. /usr/local/cuda-12.2
    lib_dir = f"{cuda_base}/lib64"
    if not os.path.isdir(lib_dir):
        lib_dir = f"{cuda_base}/targets/x86_64-linux/lib"
    if not os.path.isdir(lib_dir):
        lib_dir = "/usr/lib/x86_64-linux-gnu"

    os.environ["CPATH"] = f"{include_dir}:{os.environ.get('CPATH', '')}"
    os.environ["LIBRARY_PATH"] = f"{lib_dir}:{os.environ.get('LIBRARY_PATH', '')}"
    os.environ["LD_LIBRARY_PATH"] = f"{lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
    print(f"CUDA headers: {include_dir}")
    print(f"CUDA libs:    {lib_dir}")
else:
    print("WARNING: cuda_runtime.h not found — cublas build may fail")

# Verify
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print("nvcc:", r.stdout.split('\n')[3] if r.returncode == 0 else "not found")

In [ ]:
# 4. Clone MFI impl/cublas branch
import subprocess
%cd /content
!rm -rf mfi
!git clone -b impl/cublas --single-branch https://github.com/14NGiestas/mfi.git
%cd mfi
GIT_SHA = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"\nCloned commit: {GIT_SHA}")
!git log -1 --oneline

In [ ]:
# 5. Generate .f90 sources from .fpp/.fypp templates
%cd /content/mfi
!make clean 2>/dev/null
!make
print("\nSource generation complete")

In [ ]:
# 6. Build with fpm using the 'cublas' profile
%cd /content/mfi
print("Building with fpm --profile cublas...")
!fpm build --profile cublas 2>&1

In [ ]:
# 7. Run full test suite with GPU enabled
%cd /content/mfi
import os
os.environ["MFI_USE_CUBLAS"] = "1"

!fpm test --profile cublas 2>&1